In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.utils import resample

import numpy as np
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union
from pathlib import Path

plt.style.use("seaborn-v0_8")

PATH_IN  = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/processed/fotos_canarias.csv"
OUT_BASE = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_kmeans/"
PUNTOS_CSV = OUT_BASE + "dataset_flickr_kmeans_puntos.csv"
COMP_CSV   = OUT_BASE + "dataset_flickr_kmeans_comportamiento.csv"

import os
os.makedirs(OUT_BASE, exist_ok=True)

In [2]:
df = pd.read_csv(PATH_IN)

df = df.dropna(subset=["latitude", "longitude"]).copy()

df["datetaken"] = pd.to_datetime(df["datetaken"], errors="coerce")

df = df[df["accuracy"].fillna(0) >= 15].copy()

df = df.drop_duplicates(subset=["owner", "latitude", "longitude", "datetaken"])

In [3]:
centroides = (
    df.groupby("isla")[["latitude", "longitude"]]
      .median()
      .to_dict("index")
)

def asignar_isla_por_centroides(lat, lon, centroides):
    best, dmin = None, 1e9
    for isla, coords in centroides.items():
        d = (lat - coords["latitude"])**2 + (lon - coords["longitude"])**2
        if d < dmin:
            best, dmin = isla, d
    return best

if "isla_fix" not in df.columns:
    df["isla_fix"] = df.apply(
        lambda r: r["isla"] if pd.notna(r["isla"])
                  else asignar_isla_por_centroides(r.latitude, r.longitude, centroides),
        axis=1
    )

In [4]:
R_EARTH_M = 6371000.0

def r90_metros(group):
    lat0 = np.radians(group["latitude"].median())
    lon0 = np.radians(group["longitude"].median())
    pts  = np.radians(group[["latitude","longitude"]].values)

    d = np.arccos(np.clip(
        np.sin(lat0)*np.sin(pts[:,0]) +
        np.cos(lat0)*np.cos(pts[:,0])*np.cos(pts[:,1]-lon0),
        -1.0, 1.0
    ))
    return np.percentile(d * R_EARTH_M, 90)

In [5]:
def meters_per_degree(lat_rad):
    m_per_deg_lat = 111132.954 - 559.822 * np.cos(2*lat_rad) + 1.175 * np.cos(4*lat_rad)
    m_per_deg_lon = (111412.84 * np.cos(lat_rad)
                     - 93.5 * np.cos(3*lat_rad)
                     + 0.118 * np.cos(5*lat_rad))
    return m_per_deg_lat, m_per_deg_lon

def island_projection(df_isla):
    lat_ref = np.deg2rad(df_isla["latitude"].median())
    lon_ref = np.deg2rad(df_isla["longitude"].median())
    mlat, mlon = meters_per_degree(lat_ref)

    lat = np.deg2rad(df_isla["latitude"].values)
    lon = np.deg2rad(df_isla["longitude"].values)
    x = (lon - lon_ref) * mlon * (180/np.pi)  # (rad → °) → m
    y = (lat - lat_ref) * mlat * (180/np.pi)
    XY = np.column_stack([x, y])
    return XY

In [6]:
def candidatos_k(n):
    if n < 300:
        return list(range(2, min(10, max(3, n//30)) + 1)) or [2,3,4,5]
    elif n < 2000:
        return [3,4,5,6,8,10,12,15]
    else:
        return [5,8,10,12,15,18,20,25]

def evalua_kmeans_sub(sub_df, k):
    XY = island_projection(sub_df)

    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    labels = km.fit_predict(XY)

    if len(sub_df) > 10000:
        idx = resample(np.arange(len(sub_df)), n_samples=10000, random_state=42, replace=False)
        sil = silhouette_score(XY[idx], labels[idx]) if len(np.unique(labels[idx])) > 1 else -1.0
    else:
        sil = silhouette_score(XY, labels) if len(np.unique(labels)) > 1 else -1.0

    r90_vals = []
    for lab in np.unique(labels):
        grp = sub_df.iloc[labels == lab]
        if len(grp) >= 5:
            r90_vals.append(r90_metros(grp))
    r90_med = np.median(r90_vals) if r90_vals else np.nan

    return {
        "k": k,
        "silhouette": sil,
        "r90_med": r90_med
    }

best_by_isla = {}

for isla in sorted(df["isla_fix"].dropna().unique()):
    sub = df[df["isla_fix"] == isla][["latitude","longitude","id"]].copy()
    n = len(sub)
    if n < 50:
        continue

    resultados = [evalua_kmeans_sub(sub, k) for k in candidatos_k(n)]
    cand = pd.DataFrame(resultados)

    cand["_r90_isnan"] = cand["r90_med"].isna()
    cand = cand.sort_values(["_r90_isnan", "silhouette", "r90_med"],
                            ascending=[True, False, True]).drop(columns=["_r90_isnan"])

    best_by_isla[isla] = cand.iloc[0].to_dict()

In [7]:
df["cluster_kmeans"] = -1

for isla, params in best_by_isla.items():
    sub = df[df["isla_fix"] == isla]
    if sub.empty:
        continue

    k = int(params["k"])
    XY = island_projection(sub)

    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    labels = km.fit_predict(XY)

    df.loc[sub.index, "cluster_kmeans"] = labels

In [8]:
MIN_CLUSTER_SIZE_FINAL = 10

sizes = df[df["cluster_kmeans"] != -1].groupby("cluster_kmeans").size()
small = sizes[sizes < MIN_CLUSTER_SIZE_FINAL].index
df.loc[df["cluster_kmeans"].isin(small), "cluster_kmeans"] = -1

In [9]:
# =========================================================
# ✅ NIVELES TURÍSTICOS / LOCALES POR CLUSTER (KMEANS)
# =========================================================

def nivel_porcentaje(p):
    if p < 0.25:
        return 1
    elif p < 0.50:
        return 2
    elif p < 0.75:
        return 3
    else:
        return 4

print(" Generando niveles turístico/local por cluster (KMEANS)…")

records_niveles = []

for (isla, clus), sub in (
    df[df["cluster_kmeans"] != -1]
    .groupby(["isla_fix", "cluster_kmeans"])
):
    counts = sub["Tipo_behavior"].value_counts()

    n_fotos_turistico = int(counts.get("Turista", 0))
    n_fotos_local     = int(counts.get("Local", 0))

    p = sub["Tipo_behavior"].value_counts(normalize=True)

    p_turista = p.get("Turista", 0.0)
    p_local   = p.get("Local", 0.0)

    records_niveles.append({
        "isla_fix": isla,
        "cluster_kmeans": clus,
        "p_turista": p_turista,
        "p_local": p_local,
        "n_fotos_turistico": n_fotos_turistico,
        "n_fotos_local": n_fotos_local,
        "nivel_turistico": nivel_porcentaje(p_turista),
        "nivel_local": nivel_porcentaje(p_local)
    })

tabla_niveles = pd.DataFrame(records_niveles)

df = df.merge(
    tabla_niveles,
    on=["isla_fix", "cluster_kmeans"],
    how="left"
)

 Generando niveles turístico/local por cluster (KMEANS)…


In [10]:
df_export = df.copy()

df_export.to_csv(PUNTOS_CSV, index=False, encoding="utf-8")

if "Tipo_behavior" in df_export.columns:
    comp = (
        df_export[df_export["cluster_kmeans"] != -1]
          .groupby(["isla_fix", "cluster_kmeans"])["Tipo_behavior"]
          .value_counts(normalize=True)
          .unstack()
          .fillna(0)
    )
    comp.to_csv(COMP_CSV, encoding="utf-8")

print("✓ Exportados puntos y comportamiento correctamente en:")
print("  -", PUNTOS_CSV)
print("  -", COMP_CSV if "Tipo_behavior" in df_export.columns else "(sin comportamiento)")

✓ Exportados puntos y comportamiento correctamente en:
  - /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_kmeans/dataset_flickr_kmeans_puntos.csv
  - /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_kmeans/dataset_flickr_kmeans_comportamiento.csv


In [11]:
import pandas as pd
import geopandas as gpd
import numpy as np

from shapely.geometry import LineString
from shapely.ops import unary_union
from pathlib import Path

# =========================================================
# PATHS Y CRS
# =========================================================

PUNTOS_CSV = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_kmeans/dataset_flickr_kmeans_puntos.csv"
OUT_GPKG   = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/kmeans/kmeans_canarias.gpkg"

CRS_WGS84 = "EPSG:4326"
CRS_UTM28 = "EPSG:32628"

# =========================================================
# CARGA DATOS
# =========================================================

df = pd.read_csv(PUNTOS_CSV)

if "latitude" not in df or "longitude" not in df:
    raise ValueError("El CSV no contiene latitude/longitude")

# =========================================================
# ✅ GEO DATAFRAMES
# =========================================================

gdf_all = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=CRS_WGS84
)

gdf_points = gdf_all[gdf_all["cluster_kmeans"] != -1].copy()
gdf_noise  = gdf_all[gdf_all["cluster_kmeans"] == -1].copy()

# =========================================================
# FUNCIÓN R90
# =========================================================

def r90_m(group):
    lat0 = np.radians(group["latitude"].median())
    lon0 = np.radians(group["longitude"].median())
    pts  = np.radians(group[["latitude","longitude"]].values)

    d = np.arccos(
        np.clip(
            np.sin(lat0) * np.sin(pts[:, 0]) +
            np.cos(lat0) * np.cos(pts[:, 0]) * np.cos(pts[:, 1] - lon0),
            -1.0, 1.0
        )
    )
    return np.percentile(d * 6371000, 90)

# =========================================================
# ✅ CENTROIDES
# =========================================================

centroid_records = []

for (isla, clus), sub in gdf_points.groupby(["isla_fix", "cluster_kmeans"]):
    centroid_records.append({
        "isla_fix": isla,
        "cluster_kmeans": clus,

        # ✅ NIVELES
        "nivel_turistico": sub["nivel_turistico"].iloc[0],
        "nivel_local": sub["nivel_local"].iloc[0],

        # ✅ CONTEOS
        "n_fotos_turistico": sub["n_fotos_turistico"].iloc[0],
        "n_fotos_local": sub["n_fotos_local"].iloc[0],

        "lat": sub["latitude"].median(),
        "lon": sub["longitude"].median(),
        "n_fotos": len(sub),
        "r90_m": r90_m(sub)
    })

gdf_centroids = gpd.GeoDataFrame(
    centroid_records,
    geometry=gpd.points_from_xy(
        [c["lon"] for c in centroid_records],
        [c["lat"] for c in centroid_records]
    ),
    crs=CRS_WGS84
)

cent_utm = gdf_centroids.to_crs(CRS_UTM28)
pts_utm  = gdf_points.to_crs(CRS_UTM28)

# =========================================================
# ✅ BUFFERS R90
# =========================================================

buffers = []

for _, r in cent_utm.iterrows():
    buffers.append({
        "isla_fix": r["isla_fix"],
        "cluster_kmeans": r["cluster_kmeans"],

        "nivel_turistico": r["nivel_turistico"],
        "nivel_local": r["nivel_local"],

        "n_fotos_turistico": r["n_fotos_turistico"],
        "n_fotos_local": r["n_fotos_local"],

        "n_fotos": r["n_fotos"],
        "r90_m": r["r90_m"],
        "geometry": r.geometry.buffer(float(r["r90_m"]))
    })

gdf_buffers = gpd.GeoDataFrame(buffers, crs=CRS_UTM28).to_crs(CRS_WGS84)

# =========================================================
# ✅ HULLS
# =========================================================

hulls = []

for (isla, clus), sub in pts_utm.groupby(["isla_fix", "cluster_kmeans"]):
    hull = unary_union(sub.geometry).convex_hull

    rcent = cent_utm[
        (cent_utm["isla_fix"] == isla) &
        (cent_utm["cluster_kmeans"] == clus)
    ].iloc[0]

    hulls.append({
        "isla_fix": isla,
        "cluster_kmeans": clus,

        "nivel_turistico": rcent["nivel_turistico"],
        "nivel_local": rcent["nivel_local"],

        "n_fotos_turistico": rcent["n_fotos_turistico"],
        "n_fotos_local": rcent["n_fotos_local"],

        "n_fotos": rcent["n_fotos"],
        "r90_m": rcent["r90_m"],
        "geometry": hull
    })

gdf_hulls = gpd.GeoDataFrame(hulls, crs=CRS_UTM28).to_crs(CRS_WGS84)

# =========================================================
# ✅ SPIDER LINES
# =========================================================

spiders = []
cent_idx = {
    (r["isla_fix"], r["cluster_kmeans"]): r.geometry
    for _, r in cent_utm.iterrows()
}

for (isla, clus), sub in pts_utm.groupby(["isla_fix", "cluster_kmeans"]):
    cgeom = cent_idx[(isla, clus)]
    for _, pr in sub.iterrows():
        spiders.append({
            "isla_fix": isla,
            "cluster_kmeans": clus,
            "nivel_turistico": pr["nivel_turistico"],
            "nivel_local": pr["nivel_local"],
            "photo_id": pr.get("id", None),
            "geometry": LineString([cgeom, pr.geometry])
        })

gdf_spider = gpd.GeoDataFrame(spiders, crs=CRS_UTM28).to_crs(CRS_WGS84)

# =========================================================
# ✅ EXPORTAR GEOPACKAGE
# =========================================================

Path(OUT_GPKG).parent.mkdir(parents=True, exist_ok=True)

gdf_centroids.to_file(OUT_GPKG, layer="kmeans_centroids",    driver="GPKG")
gdf_buffers.to_file(  OUT_GPKG, layer="kmeans_buffers_r90", driver="GPKG")
gdf_hulls.to_file(    OUT_GPKG, layer="kmeans_hulls",       driver="GPKG")
gdf_points.to_file(   OUT_GPKG, layer="kmeans_points",     driver="GPKG")
gdf_noise.to_file(    OUT_GPKG, layer="kmeans_noise",      driver="GPKG")
gdf_spider.to_file(   OUT_GPKG, layer="kmeans_spider",     driver="GPKG")

print("✅ Exportado GeoPackage KMEANS con niveles turístico/local:", OUT_GPKG)

✅ Exportado GeoPackage KMEANS con niveles turístico/local: /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/kmeans/kmeans_canarias.gpkg
